# Supervised Fine-Tuning (SFT) of NileChat-3B

Fine-tunes `UBC-NLP/NileChat-3B` on Ahmed's real WhatsApp conversations (chat-format JSONL) with Unsloth + LoRA + `SFTTrainer`, on Google Colab with checkpoints on Google Drive.

**Result:** the best validation loss (~1.625) is reached around **step 900–1200**, and the model overfits from about step 1550–1650, well before the full 2-epoch / 3116-step schedule. Training was stopped at **checkpoint-1000**. Section 8 merges that checkpoint into the standalone model (`ahmed_twin/model_merged_nilechat`) that the DPO and evaluation stages build on.

**Known limitation (kept as-is, because it's the configuration that produced the model):** `save_steps=500` + `save_total_limit=2` keeps only the 2 newest checkpoints. Left unattended past step ~2000, training would delete checkpoint-1000. It was stopped manually before that. (The DPO stage fixes this with best-model tracking and early stopping.)

## Contents
1. Setup: Drive, GPU, dependencies, Hugging Face login
2. Load the base model and tokenizer
3. Data: load and format the WhatsApp JSONL, tokenizer sanity check
4. LoRA adapters
5. Train: `SFTTrainer` config, optional cleanup of an old run, disk check, training
6. Inference check
7. Save the merged model (only if training ran to completion)
8. **Recover checkpoint-1000** (what this project actually used)

**Before committing: Edit → Clear all outputs.** Outputs contain real messages.

## 1. Setup
### 1.1 Mount Drive and confirm the GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
assert torch.cuda.is_available(), "No GPU -- Runtime > Change runtime type > GPU"
print("GPU:", torch.cuda.get_device_name(0))

### 1.2 Install dependencies

In [ ]:
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes

### 1.3 Imports and Hugging Face login (NileChat-3B is a gated model)

`unsloth` is imported first so its optimizations apply to `transformers`/`trl`. The token is entered at runtime with `getpass`, never stored in the notebook.

In [ ]:
import unsloth  # must come before transformers / trl / peft
from unsloth import FastLanguageModel, is_bfloat16_supported

import os, json, random, shutil
from huggingface_hub import login
from getpass import getpass

login(token=getpass("Hugging Face token: "))

MODEL_NAME = "UBC-NLP/NileChat-3B"
DRIVE_ROOT = "/content/drive/MyDrive"
TRAIN_PATH = f"{DRIVE_ROOT}/ahmed_twin/train.jsonl"
VAL_PATH = f"{DRIVE_ROOT}/ahmed_twin/validation.jsonl"
CHECKPOINT_DIR = f"{DRIVE_ROOT}/ahmed_twin/checkpoints_nilechat"
MERGED_MODEL_PATH = f"{DRIVE_ROOT}/ahmed_twin/model_merged_nilechat"   # loaded by the DPO and evaluation stages
MAX_SEQ_LENGTH = 2048


# Used by Sections 7 and 8. Defined here so Section 8 also works in a fresh session (Section 1 + 8).
def verify_merged(path):
    # Check real file sizes, not just that the folder exists: a fp16 3B model is several GB.
    files = os.listdir(path)
    total_gb = sum(os.path.getsize(os.path.join(path, f)) for f in files) / 1e9
    weights = [f for f in files if f.endswith(".safetensors") and "index" not in f]
    print(f"{path}: {total_gb:.2f} GB, weight shards: {weights}")
    assert total_gb > 3, f"Only {total_gb:.2f} GB -- the save failed (disk space, quota, or interrupted upload)."
    assert weights, "No .safetensors weight shards -- only an index/config was saved. Check Drive quota and re-run."
    print("Verified: merged model is complete.")

## 2. Load the base model and tokenizer

Loaded **before** the data, because formatting the data uses the tokenizer's chat template.

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
)
print("Loaded:", MODEL_NAME)

## 3. Data
### 3.1 Load and format the dataset

Each record is a multi-turn chat (`system` / `user` / `assistant`). The tokenizer's chat template turns it into the exact text format the model expects.

In [ ]:
from datasets import Dataset


def load_chat_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return Dataset.from_list([json.loads(line) for line in f])


def format_chat(example):
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)
    return {"text": text}


train_dataset = load_chat_jsonl(TRAIN_PATH).map(format_chat)
val_dataset = load_chat_jsonl(VAL_PATH).map(format_chat)

print(f"train: {len(train_dataset)} | validation: {len(val_dataset)}")
print(train_dataset[0]["text"][:300])

### 3.2 Tokenizer sanity check on real data

NileChat continued pretraining on top of the Qwen2.5 architecture, so its tokenizer is worth checking on real messages. A low characters-per-token ratio would mean Arabic is being split into many small pieces.

In [ ]:
with open(TRAIN_PATH, encoding="utf-8") as f:
    all_lines = [json.loads(l) for l in f]

for ex in random.Random(0).sample(all_lines, 5):
    text = tokenizer.apply_chat_template(ex["messages"], tokenize=False)
    ids = tokenizer(text)["input_ids"]
    print(f"chars={len(text)} tokens={len(ids)} ratio={len(text) / len(ids):.2f}")
    print(text[:200])
    print("---")

## 4. LoRA adapters

Rank 32 on every attention and MLP projection. Only these small adapter matrices are trained; the 3B base weights stay frozen.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=32,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "up_proj", "down_proj", "o_proj", "gate_proj"],
)

## 5. Train
### 5.1 Training configuration

The exact configuration that produced the project's SFT model: effective batch size 16 (2 × 8 accumulation), LR 3e-4 with linear decay, packing on, evaluation every 50 steps.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=True,
    args=TrainingArguments(
        learning_rate=3e-4,
        lr_scheduler_type="linear",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        num_train_epochs=2,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        eval_strategy="steps",
        eval_steps=50,
        optim="adamw_8bit",
        weight_decay=0.01,
        warmup_steps=10,
        output_dir=CHECKPOINT_DIR,
        save_strategy="steps",
        save_steps=500,
        save_total_limit=2,  # bounded Drive usage -- see the known limitation at the top
        seed=0,
        report_to="none",
    ),
)

### 5.2 Optional: delete an old, broken run before starting fresh

**Off by default.** This deletes the whole checkpoints folder, **including checkpoint-1000**. Only set it to `True` when you're deliberately starting a brand-new training run and have nothing in that folder you need.

In [ ]:
DELETE_OLD_CHECKPOINTS = False

if DELETE_OLD_CHECKPOINTS:
    shutil.rmtree(CHECKPOINT_DIR, ignore_errors=True)
    print("Deleted:", CHECKPOINT_DIR)
else:
    print("Kept existing checkpoints in", CHECKPOINT_DIR)

### 5.3 Pre-flight disk-space check

In [ ]:
free_gb = shutil.disk_usage(DRIVE_ROOT).free / 1e9
print(f"Free space on Drive: {free_gb:.2f} GB")
assert free_gb > 10, (
    f"Only {free_gb:.2f} GB free -- checkpoints plus the merged model need 6-10 GB. "
    "Free up space first, or checkpoints may silently fail to save (as happened previously)."
)

### 5.4 Train

Watch: training loss going down, validation loss following it (not rising), grad norm stable. **This project stopped training at checkpoint-1000** (Runtime → Interrupt execution) once validation loss bottomed out.

In [ ]:
trainer.train()

## 6. Inference check

Generation settings from the project's decoding sweep: `temperature=0.4`, repetition penalty, and **an explicit end-of-turn token**. Without the end-of-turn token, the model keeps going and invents extra turns. These are the same three prompts used to compare checkpoints.

In [ ]:
FastLanguageModel.for_inference(model)

SYSTEM_MESSAGE = ("Continue the conversation as Ahmed. Match the language, tone, and context of the "
                  "conversation. Do not claim to know facts that are not supported by the dialogue.")
EOS_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")
test_prompts = ["احكيلي عن نفسك", "معلش هو انت فاضي دلوقتي؟", "عامل ايه النهاردة؟"]

for p in test_prompts:
    prompt = tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM_MESSAGE}, {"role": "user", "content": p}],
        tokenize=False, add_generation_prompt=True,
    )
    enc = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to("cuda")
    out = model.generate(**enc, max_new_tokens=100, do_sample=True, temperature=0.4,
                         repetition_penalty=1.1, no_repeat_ngram_size=4,
                         eos_token_id=EOS_ID, pad_token_id=EOS_ID)
    print(p, "->")
    print(tokenizer.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip())
    print("---")

## 7. Save the merged model (only if training ran to completion)

Skip this section if you stopped training early. This project did, so it uses Section 8 instead. Both sections write to the same `MERGED_MODEL_PATH`, which is what the DPO and evaluation stages load.

In [ ]:
model.save_pretrained_merged(MERGED_MODEL_PATH, tokenizer, save_method="merged_16bit")
verify_merged(MERGED_MODEL_PATH)

## 8. Recover checkpoint-1000 (what this project used)

Training was stopped at **step 1000**, inside the best-validation-loss window (~900–1200) and before overfitting starts (~1550–1650). This section merges that checkpoint into a standalone model.

It works in a **fresh session** too: run Section 1 (setup + login), then this section. The checkpoint folder is loaded directly with Unsloth, which reads the base model's name from the adapter's config. Loading it through `peft.PeftModel` instead makes Unsloth's merge fail; that happened in the DPO stage.

In [ ]:
import gc
CHECKPOINT_TO_RECOVER = f"{CHECKPOINT_DIR}/checkpoint-1000"
assert os.path.exists(os.path.join(CHECKPOINT_TO_RECOVER, "adapter_config.json")), (
    f"No adapter found at {CHECKPOINT_TO_RECOVER} -- was it deleted (save_total_limit) or never written?")

# Free the training model first: two fp16 3B models don't fit on a T4 together.
for name in ("trainer", "model"):
    globals().pop(name, None)
gc.collect()
torch.cuda.empty_cache()

recovered_model, recovered_tokenizer = FastLanguageModel.from_pretrained(
    model_name=CHECKPOINT_TO_RECOVER, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=False,
)
recovered_model.save_pretrained_merged(MERGED_MODEL_PATH, recovered_tokenizer, save_method="merged_16bit")
verify_merged(MERGED_MODEL_PATH)